<a href="https://colab.research.google.com/github/WhoisMonesh/Colab-Course-Downloader/blob/main/Colab-Course-Downloader.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Course Downloader

Download courses from YouTube, Coursera, and Udemy to Google Drive.


### 1. Mount Google Drive

In [0]:
from google.colab import drive
drive.mount('/content/drive')

### 2. Install Dependencies

In [0]:
!pip install -U yt-dlp curl_cffi -q
print('Dependencies installed.')

### 3. Configuration

In [0]:
# === CONFIGURATION ===
SAVE_PATH = '/content/downloads/CourseDownloader/'  # local temp dir
DRIVE_PATH = '/content/drive/My Drive/CourseDownloader/'  # final destination

COURSE_URL = ''  # Course URL (YouTube playlist, Coursera, or Udemy)
FORMAT = 'bestvideo[height<=720]+bestaudio/best[height<=720]'  # Video format
AUDIO_ONLY = False  # Extract audio only (MP3)
MAX_VIDEOS = 0  # Max videos to download (0 = all)

# Optional: authentication for paid platforms (Coursera/Udemy)
USERNAME = ''
PASSWORD = ''

KEEP_ALIVE = True  # prevent Colab timeout

# === END CONFIGURATION ===

### 4. Download

In [0]:
import os, time, shutil
from google.colab import files

def format_bytes(n):
    for u in ['B', 'KB', 'MB', 'GB', 'TB']:
        if n < 1024: return f'{n:.1f} {u}'
        n /= 1024
    return f'{n:.1f} PB'

def get_all_files(root):
    result = []
    for dirpath, _, filenames in os.walk(root):
        for f in filenames:
            result.append(os.path.join(dirpath, f))
    return result

def main():
    import yt_dlp
    from IPython.display import display, HTML, Javascript
    try:
        from yt_dlp.networking.impersonate import ImpersonateTarget
        try:
            IMPERSONATE = ImpersonateTarget(client='chrome')
        except TypeError:
            IMPERSONATE = ImpersonateTarget(target='chrome')
    except Exception:
        IMPERSONATE = None
    if KEEP_ALIVE:
        display(Javascript('''
            function keepAlive(){
                var btn=document.querySelector("colab-connect-button");
                if(btn)btn.click()
            }
            setInterval(keepAlive,120000);
        '''))
        print('Keep-alive active')

    if not COURSE_URL:
        print('ERROR: Set COURSE_URL in config')
        return

    begin = time.time()
    os.makedirs(SAVE_PATH, exist_ok=True)
    print(f'Save path: {SAVE_PATH} (local)')

    # Detect platform and set auth
    url_lower = COURSE_URL.lower()
    needs_auth = 'coursera.org' in url_lower or 'udemy.com' in url_lower
    cookies_path = None

    if needs_auth:
        print('Upload cookies.txt (export from browser using a cookies extension)...')
        uploaded = files.upload()
        if uploaded:
            cookies_path = list(uploaded.keys())[0]
            print(f'Cookies loaded: {cookies_path}')
    platform = 'Coursera' if 'coursera.org' in url_lower else 'Udemy' if 'udemy.com' in url_lower else 'YouTube'
    print(f'Platform: {platform}')

    ydl_opts = {
        'outtmpl': os.path.join(SAVE_PATH, '%(playlist_title|-)s/%(playlist_index|0)02d - %(title)s.%(ext)s'),
        'format': FORMAT,
        'quiet': True,
        'no_warnings': True,
        'playlistend': MAX_VIDEOS if MAX_VIDEOS > 0 else None,
        'embedmetadata': True,
        'embedchapters': True,
    }
    if IMPERSONATE:
        ydl_opts["impersonate"] = IMPERSONATE
    if AUDIO_ONLY:
        ydl_opts['format'] = 'bestaudio/best'
        ydl_opts['postprocessors'] = [{
            'key': 'FFmpegExtractAudio',
            'preferredcodec': 'mp3',
            'preferredquality': '192',
        }]
    if cookies_path:
        ydl_opts['cookiefile'] = cookies_path
        ydl_opts['password'] = PASSWORD

    # Detect course info
    detect_opts = {'quiet': True, 'no_warnings': True, 'extract_flat': True}
    if IMPERSONATE:
        detect_opts["impersonate"] = IMPERSONATE
    if cookies_path:
        detect_opts['cookiefile'] = cookies_path

    with yt_dlp.YoutubeDL(detect_opts) as ydl:
        info = ydl.extract_info(COURSE_URL, download=False)

    course_title = info.get('title', 'Course')
    total = 0
    if 'entries' in info:
        entries = list(info.get('entries', []))
        total = len(entries)
        if MAX_VIDEOS > 0:
            total = min(total, MAX_VIDEOS)
        print(f'Course: {course_title} ({total} lectures)')
    else:
        print(f'Lecture: {info.get("title", "Unknown")}')

    progress_display = display(HTML('<pre>Starting...</pre>'), display_id='dl-progress')

    def progress_hook(d):
        if d['status'] != 'downloading':
            return
        info_dict = d.get('info_dict', {})
        idx = info_dict.get('playlist_index', '')
        total_pl = info_dict.get('playlist_count', 0)
        pct = d.get('_percent_str', '').strip()
        speed = d.get('_speed_str', '').strip()
        downloaded = d.get('downloaded_bytes', 0)
        total_bytes = d.get('total_bytes', 0) or d.get('total_bytes_estimate', 0)
        bar_len = int(downloaded * 50 // max(total_bytes, 1)) if total_bytes else 0
        bar = '#' * bar_len + '-' * (50 - bar_len)
        msg = 'Downloading'
        if total_pl and idx:
            msg += f' ({idx}/{total_pl})'
        msg += f': |{bar}| {pct}'
        if speed:
            msg += f' @ {speed}'
        progress_display.update(HTML(f'<pre>{msg}</pre>'))

    ydl_opts['progress_hooks'] = [progress_hook]
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.download([COURSE_URL])

    end = time.time()
    elapsed = int(end - begin)
    print()
    print('=' * 50)
    print('COMPLETE')
    print(f'Elapsed: {elapsed // 60}m {elapsed % 60}s')

    items = get_all_files(SAVE_PATH)
    total_sz = sum(os.path.getsize(f) for f in items)
    print(f'Downloaded {len(items)} files ({format_bytes(total_sz)})')

    if len(items) > 1:
        import zipfile
        processed = 0
        zpath = SAVE_PATH.rstrip('/').rstrip('\\') + '.zip'
        print(f'\nZipping {len(items)} files...')
        with zipfile.ZipFile(zpath, 'w', zipfile.ZIP_DEFLATED) as zf:
            for fp in items:
                arcname = os.path.relpath(fp, SAVE_PATH)
                zf.write(fp, arcname)
                processed += os.path.getsize(fp)
                pct = int(processed * 100 / total_sz) if total_sz else 0
                bar = '#' * (pct // 2) + '-' * (50 - pct // 2)
                progress_display.update(HTML(f'<pre>Zipping: |{bar}| {pct}% | {arcname}</pre>'))
        print(f'Zip: {os.path.basename(zpath)} ({format_bytes(os.path.getsize(zpath))})')
        display(HTML(f'<a href="{zpath}" download>Download ZIP</a>'))
        files.download(zpath)

    print(f'\nMoving to Drive...')
    os.makedirs(DRIVE_PATH, exist_ok=True)
    for item in os.listdir(SAVE_PATH):
        s = os.path.join(SAVE_PATH, item)
        d = os.path.join(DRIVE_PATH, item)
        if os.path.isdir(s):
            if os.path.exists(d):
                shutil.rmtree(d)
            shutil.move(s, DRIVE_PATH)
        else:
            shutil.move(s, d)
    print(f'Final: {DRIVE_PATH}')
    for item in sorted(os.listdir(DRIVE_PATH)):
        if os.path.isdir(os.path.join(DRIVE_PATH, item)):
            print(f'  {item}/')
        else:
            print(f'  {item}')
    print('=' * 50)

if __name__ == '__main__':
    main()